# Знания модели и поиск информации

Языковая модель умеет работать с текстом и отвечать на вопросы, опираясь на знания, полученные во время обучения. Однако это не означает, что модель знает всё, что может понадобиться в конкретной задаче.

Иногда нужная информация относится к предметной области, которая недостаточно представлена в обучающих данных. Например, модель может хорошо знать общие сведения о медицине, истории или программировании, но не знать внутренние правила конкретной организации.

В других случаях информация может быть просто новой. Документ мог быть создан после обучения модели, а сведения о проекте могли измениться с тех пор, как модель была обучена.

Наконец, нужные сведения могут быть специфическими для конкретного источника. Это могут быть внутренние названия, результаты экспериментов, история изменений проекта, содержание отдельного набора документов или принятые внутри команды правила.

При этом проблема не обязательно заключается в том, что модель «ничего не знает» о нужной области. Иногда у неё есть общие знания, но конкретный источник содержит сведения, которые важнее для нашей задачи. Они могут уточнять общую картину, дополнять её или даже отличаться от того, что обычно считается верным.

Получается важное различие: модель может уметь хорошо работать с вопросом как с текстовой задачей, но при этом не иметь доступа именно к тем сведениям, на которых должен основываться ответ.

Например, мы уже использовали языковую модель для поиска ошибок в предложении. Представим теперь, что такую систему хотят использовать в школе. Причём не просто для проверки русского языка «вообще», а, например, для работы с учениками определённого класса. Нужно учитывать программу, по которой они учатся, и требования, предъявляемые к их работам. Какие материалы могли бы понадобиться модели в таком случае?

Вероятно, одного текста задания будет недостаточно. Можно было бы использовать учебники, материалы курса, ФОСы и другие документы, по которым определяется, что именно должен знать и уметь ученик. Причём таких материалов может быть много: несколько учебников, наборы заданий, методические материалы, документы разных лет.

А если та же идея используется для другого предмета, набор источников и критерии проверки уже будут другими.

Получается, что нам иногда недостаточно общих знаний модели. Нужно дать ей возможность находить нужную информацию среди **большой коллекции конкретных источников**, не передавая всю коллекцию в каждый запрос.

Другая ситуация: представим систему, которая используется внутри команды. В процессе работы в ней накапливаются описания компонентов системы, результаты работы, тестовые сценарии, история изменений и другие документы.

Со временем таких материалов становится много. Участник команды может спросить, например, что умеет определённый компонент, когда исправили какую-то проблему или почему конкретный случай получил определённый статус. Ответ на такой вопрос может находиться в одном из документов или складываться из информации из нескольких документов.

Здесь возникает уже знакомая нам проблема: как быстро найти среди большой коллекции материалов те фрагменты, которые относятся к заданному вопросу?

### Использование поиска по коллекции

Если у нас есть большая коллекция материалов, по ней можно организовать поиск. Например, по вопросу участника команды можно попробовать найти в документах фрагменты, которые относятся к нужной ситуации. Мы уже умеем использовать для такого поиска эмбеддинги: сравнивать представление запроса с представлениями текстовых фрагментов и находить наиболее близкие по смыслу.

Но что делать с найденными фрагментами дальше? Поиск может показать несколько подходящих фрагментов из разных документов. Однако сам по себе он не сформирует ответ на вопрос и, если нужная информация распределена между несколькими источниками, не объединит её в связное объяснение.

Значит, одного поиска недостаточно: нам нужно каким-то образом передать найденную информацию языковой модели, чтобы она использовала её при формировании ответа.

### Retrieval-Augmented Generation

Мы не первые столкнулись с этой задачей. Подход, в котором языковая модель использует информацию, предварительно найденную во внешней коллекции данных, получил название **Retrieval-Augmented Generation (RAG)**.

Само название описывает последовательность действий:

- **Retrieval** — найти в коллекции информацию, релевантную текущему запросу;
- **Augmented** — дополнить запрос или контекст модели найденной информацией;
- **Generation** — сформировать ответ с учётом этого контекста.

Таким образом, языковая модель получает возможность использовать сведения, которых может не быть среди её собственных знаний.

### Пример: система «221B»

Рассмотрим небольшую коллекцию документов, относящихся к внутренней системе «221B», которая используется некоей командой для анализа событий и расследования инцидентов.

Команда предоставила нам следующие материалы:

- `overview.md` — общее описание системы;
- `concept.md` — основные концепции и роли компонентов;
- `beta-release.md` — описание бета-версии;
- `case-archive.md` — архив расследований;
- `moriarty-tests.md` — тестовые сценарии;
- `git-history.txt` — история изменений;
- `april-fools.md` — шуточная статья о системе.

Эти документы содержат информацию, которая нужна участнику команды, но не является частью общих знаний языковой модели.

Попробуем построить систему, которая сможет отвечать на вопросы о 221B, находя нужную информацию в этой коллекции. Для начала посмотрим, с какими данными нам предстоит работать.

In [112]:
from pathlib import Path
from IPython.display import display, Markdown

data_dir = Path("data")

files = sorted(data_dir.iterdir())

overview = data_dir / "overview.md"

print("=" * 80)
print(overview.name)
print("=" * 80)

display(Markdown(overview.read_text(encoding="utf-8")))

overview.md


# Система 221B

221B — система для анализа событий и расследования инцидентов. Она состоит из нескольких компонентов, выполняющих разные функции.

## Основные компоненты

**Holmes** анализирует поток событий и обнаруживает подозрительные закономерности, которые могут указывать на инцидент.

**Watson** получает результаты работы Holmes и формирует объяснение обнаруженного случая.

**Lestrade** проверяет результаты анализа и определяет, требует ли случай дополнительного рассмотрения.

**Mrs. Hudson** ведёт архив расследований и хранит сведения о ранее рассмотренных случаях.

**Moriarty** используется для создания контролируемых тестовых случаев, на которых проверяется работа системы.

Таким образом, в обычном сценарии Holmes обнаруживает подозрительную ситуацию, Watson анализирует её, а Lestrade проверяет полученный результат. Архив Mrs. Hudson предоставляет дополнительную информацию о прошлых случаях, а Moriarty используется для тестирования системы.

In [133]:
for file in files:
    if file.name == "overview.md":
        continue

    text = file.read_text(encoding="utf-8")

    print("=" * 80)
    print(file.name + '\n')

    print(text[:500] + "...\n")

april-fools.md

# Мы заменили Holmes на Moriarty — и система стала работать на 146% лучше

*1 апреля 2026 года*

Команда 221B рада представить самое важное обновление за всю историю проекта: теперь обнаружением инцидентов занимается Moriarty.

Решение было принято после нескольких месяцев наблюдений за Holmes. Несмотря на выдающиеся результаты, Holmes продолжал придерживаться устаревшего подхода: сначала находить закономерность, а уже потом пытаться понять, что произошло. Moriarty предложил более современную ст...

beta-release.md

# 221B Beta 0.8

После нескольких месяцев разработки команда 221B выпустила первую публичную бета-версию системы.

Главное изменение Beta 0.8 — переход от экспериментов с отдельными компонентами к единому процессу расследования инцидентов.

## Что нового

**Holmes** получил поддержку паттернов `RED` и `BLUE`. Теперь система может автоматически обнаруживать эти типы аномалий в потоке событий.

**Watson** научился формировать объяснения для случаев, обнаруженн

Мы видим, что документы отличаются не только форматом и содержанием, но и характером информации. Например, `april-fools.md` содержит шуточный материал, поэтому пока не совсем понятно, можно ли использовать его наравне с остальными документами как источник фактической информации.

Остальные документы выглядят как материалы, на которые можно непосредственно опираться при ответах на вопросы о системе.

In [134]:
excluded_files = {"april-fools.md"}

documents = []
for file in sorted(data_dir.iterdir()):
    if file.is_file() and file.name not in excluded_files:
        text = file.read_text(encoding="utf-8")
        documents.append({
            "filename": file.name,
            "text": text,
        })

print(f"Загружено документов: {len(documents)}")

for doc in documents:
    print(f"{doc['filename']}: {len(doc['text'])} символов")

Загружено документов: 6
beta-release.md: 1143 символов
case-archive.md: 967 символов
concept.md: 1654 символов
git-history.txt: 704 символов
moriarty-tests.md: 997 символов
overview.md: 963 символов


Прежде чем искать информацию, нужно подготовить нашу коллекцию документов к поиску. Даже в нашем небольшом примере не стоит передавать модели весь корпус при каждом запросе. Если вопрос касается, например, истории изменений, нет необходимости включать в контекст полное описание системы, архив расследований и тестовые сценарии. В реальной коллекции документов такой подход быстро становится неэффективным и по объёму контекста, и по количеству нерелевантной информации.

#### Подготовка фрагментов

Чанкинг нам уже знаком: прежде чем искать информацию по коллекции, документы обычно разбивают на части, которые становятся отдельными единицами поиска.

Здесь мы не будем подбирать сложную стратегию. Наши документы небольшие и имеют достаточно простую структуру, поэтому для демонстрации будем считать каждый абзац отдельным фрагментом.

В реальной системе способ разбиения зависит от структуры документов и может существенно влиять на качество поиска.

In [135]:
chunks = []
for doc in documents:
    paragraphs = [
        p.strip()
        for p in doc["text"].split("\n\n")
        if p.strip() and any(map(str.isalpha, p))
    ]

    for i, paragraph in enumerate(paragraphs):
        chunks.append({
            "filename": doc["filename"],
            "chunk_id": i,
            "text": paragraph,
        })

print(f"Всего фрагментов: {len(chunks)}")

Всего фрагментов: 45


In [136]:
import random

for chunk in random.sample(chunks, 5):
    print("=" * 80)
    print(f"{chunk['filename']} | chunk {chunk['chunk_id']}")
    print(chunk["text"])

overview.md | chunk 7
**Moriarty** используется для создания контролируемых тестовых случаев, на которых проверяется работа системы.
git-history.txt | chunk 0
2026-03-14  add Pattern-GOLD detection
Holmes теперь обнаруживает Pattern-GOLD после четырёх последовательных событий. Новый детектор отключён в стандартном процессе Beta 0.8.
concept.md | chunk 1
В системах анализа событий часто возникает соблазн объединить обнаружение подозрительной ситуации и её интерпретацию в один этап. 221B использует другой подход: сначала система должна обнаружить закономерность в данных, а затем отдельно объяснить, что она может означать.
git-history.txt | chunk 4
2026-04-01  archive experimental GOLD cases
Экспериментальные случаи GOLD теперь сохраняются в архиве со статусом EXPERIMENTAL.
beta-release.md | chunk 6
**Lestrade** теперь может автоматически проверять результаты Watson для паттернов `RED` и `BLUE`.


Некоторые фрагменты получились очень короткими и сами по себе малоинформативными. Также разбиение не всегда сохраняет структуру документа: заголовок может оказаться отдельно от текста, к которому он относится.

Для реальной системы чанкинг можно сделать более осмысленным: учитывать структуру документа и связывать заголовки с соответствующим содержимым и т.п. В нашем демо пока оставим этот вариант без улучшений.

### Представление фрагментов

Теперь у нас есть набор фрагментов, которые могут стать единицами поиска. Чтобы искать среди них по смыслу, представим каждый фрагмент в виде эмбеддинга. Тогда вопрос пользователя мы сможем представить таким же способом и сравнить его с фрагментами коллекции. Механизм нам уже знаком, поэтому здесь сразу применим его к подготовленным данным.

In [137]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
)

print(chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(45, 384)


### Поиск по эмбеддингам

Теперь осталось связать запрос пользователя с нашей коллекцией фрагментов.

Для этого представим сам запрос в виде эмбеддинга и сравним его с эмбеддингами всех фрагментов. Поскольку эмбеддинги нормализованы, скалярное произведение можно использовать как меру близости: чем выше значение, тем ближе фрагмент к запросу.

In [121]:
import numpy as np


def retrieve(query, top_k=3):
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True,
    )
    scores = chunk_embeddings @ query_embedding
    top_indices = np.argsort(scores)[-top_k:][::-1]

    return [
        {
            **chunks[i],
            "score": float(scores[i]),
        }
        for i in top_indices
    ]

In [122]:
test_question = "Что делает система 221B?"

for result in retrieve(test_question):
    print("=" * 60)
    print(f"{result['filename']} | score={result['score']:.3f}")
    print(result["text"])

overview.md | score=0.775
# Система 221B
beta-release.md | score=0.740
После нескольких месяцев разработки команда 221B выпустила первую публичную бета-версию системы.
overview.md | score=0.689
221B — система для анализа событий и расследования инцидентов. Она состоит из нескольких компонентов, выполняющих разные функции.


Начали с простого общего вопроса: «Что делает система 221B?» Нужная информация в коллекции есть: третий найденный фрагмент прямо описывает назначение системы. Но два первых результата сами по себе не дают ответа на вопрос, хотя получили достаточно высокие оценки близости.

Это важное свойство семантического поиска: он ищет фрагменты, близкие к запросу по смыслу, но не проверяет, содержит ли найденный текст полный ответ на вопрос.

Посмотрим, как поиск поведёт себя на более конкретных запросах.

In [123]:
test_questions = [
    "Какую роль играет Holmes в системе 221B?",
    "Что произошло с Case-21 и почему он получил статус REVIEW?",
    "Какие паттерны ловит Холмс в бете?",
]

for question in test_questions:
    print("=" * 80)
    print(question)

    results = retrieve(question)
    for result in results:
        print(
            f"\n{result['filename']} | "
            f"score={result['score']:.3f}"
        )
        print(result["text"])

Какую роль играет Holmes в системе 221B?

beta-release.md | score=0.730
После нескольких месяцев разработки команда 221B выпустила первую публичную бета-версию системы.

overview.md | score=0.718
**Holmes** анализирует поток событий и обнаруживает подозрительные закономерности, которые могут указывать на инцидент.

overview.md | score=0.674
221B — система для анализа событий и расследования инцидентов. Она состоит из нескольких компонентов, выполняющих разные функции.
Что произошло с Case-21 и почему он получил статус REVIEW?

case-archive.md | score=0.546
## Case-21 — Blue Window
Дата: 2025-03-17
Holmes обнаружил паттерн `BLUE`.
Watson сформировал объяснение, но не указал идентификатор источника.
Lestrade отклонил объяснение и перевёл дело в статус `REVIEW`.

moriarty-tests.md | score=0.515
## Test Case B — Blue Window
Сценарий содержит события, соответствующие условиям паттерна `BLUE`, но не содержит идентификатора источника. Ожидается, что Holmes обнаружит `BLUE`, а Lestrade перевед

Поиск уже работает: для каждого вопроса мы получаем наиболее близкие по смыслу фрагменты и оценку их близости. Но сразу видно, что семантическая близость не гарантирует, что найденный фрагмент действительно содержит ответ на вопрос.

Иногда в результат попадает общий или лишь частично относящийся к вопросу текст. В других случаях нужная информация есть в коллекции, но её фрагмент не оказывается среди первых результатов.

Для нашего небольшого демонстрационного корпуса это уже хорошо видно на вопросах о Holmes.

Пока ничего не будем исправлять. Посмотрим, что произойдёт, если найденные фрагменты передать языковой модели и попросить её сформировать ответ.

### Подключаем языковую модель

Поиск уже умеет находить фрагменты, но сам по себе не формирует ответ. Теперь подключим языковую модель, которая будет работать с найденным контекстом.

В нашем случае модель запускается локально через Ollama, а обращаться к ней мы будем через совместимый с OpenAI API интерфейс, как в предыдущем демо.

Сначала попробуем обратиться к ней напрямую, без поиска по коллекции.

In [31]:
from openai import OpenAI

client = OpenAI(
    base_url="http://ollama:11434/v1",
    api_key="ollama",
)

response = client.chat.completions.create(
    model="qwen3:4b-instruct",
    messages=[
        {
            "role": "user",
            "content": test_question,
        }
    ],
)

print(response.choices[0].message.content)

Система 221B — это, вероятно, неправильное или неопределённое обозначение. На текущий момент в науке, технике или общественной практике нет известной системы под названием "221B". Возможно, вы имеете в виду:

1. **Код 221B в системе адресации или классификации** — например, в системе адресов, номерах, или кодах, но даже тогда такой код не имеет общепринятого значения.

2. **Система 221B как отсылка к "221B Baker Street"** — это известный адрес в Лондоне, где живёт Шерлок Холмс. Это литературное образование, а не реальная система.

3. **Опечатка или неправильная интерпретация** — вы могли иметь в виду, например, "система 2.2.1.B" или что-то другое.

Если вы имели в виду какую-то конкретную систему (например, в информатике, энергетике, безопасности, медицине), пожалуйста, уточните контекст.

**Вывод**: Система 221B не существует в каких-либо официальных или широко признанных системах. Возможно, вы ошиблись в названии или имели в виду что-то другое. Уточните, пожалуйста, контекст или цель

Без дополнительного контекста модель не знает нашу внутреннюю систему и пытается интерпретировать знакомое ей обозначение `221B` через собственные знания. В результате получается подробный, но не имеющий отношения к нашей коллекции ответ.

Теперь используем результаты поиска по назначению: передадим найденные фрагменты языковой модели вместе с вопросом.

Важно явно указать модели, что эти фрагменты являются источником ответа. Если в переданном контексте нужной информации нет, модель должна сообщить об этом, а не дополнять ответ своими знаниями.

In [61]:
def build_context(question, top_k=3):
    results = retrieve(question, top_k=top_k)
    context = "\n\n".join(
        f"[{r['filename']}]\n{r['text']}"
        for r in results
    )
    return context, results


def ask_rag(question, context=None, top_k=3):
    if context is None:
        context, _ = build_context(question, top_k=top_k)

    response = client.chat.completions.create(
        model="qwen3:4b-instruct",
        messages=[
            {
                "role": "system",
                "content": (
                    "Отвечай на вопрос только на основе переданного контекста. "
                    "Если в контексте нет ответа, так и скажи."
                ),
            },
            {
                "role": "user",
                "content": f"Контекст:\n\n{context}\n\nВопрос: {question}",
            },
        ],
    )
    return response.choices[0].message.content

In [60]:
print(ask_rag(test_question))

Система 221B делает анализ событий и расследование инцидентов.


После добавления найденного фрагмента ответ становится коротким и предметным. Это связано не только с тем, что модель получила нужную информацию, но и с тем, что предоставленного контекста достаточно лишь для краткого ответа, а инструкция запрещает дополнять его собственными знаниями и догадками.

При этом сама языковая модель не изменилась — мы не дообучали её и не добавляли новые знания в её параметры. Изменился только контекст, который мы передали ей перед генерацией ответа.

In [66]:
for question in test_questions:
    print("=" * 80)
    print(question + '\n')
    print(ask_rag(question))

Какую роль играет Holmes в системе 221B?

Holmes анализирует поток событий и обнаруживает подозрительные закономерности, которые могут указывать на инцидент.
Что произошло с Case-21 и почему он получил статус REVIEW?

С Case-21 произошло следующее: Holmes обнаружил паттерн `BLUE`. Watson сформировал объяснение, но не указал идентификатор источника. Lestrade отклонил объяснение и перевёл дело в статус `REVIEW`.

Таким образом, Case-21 получил статус `REVIEW`, потому что Watson не указал идентификатор источника в своём объяснении, что привело к его отклонению со стороны Lestrade.
Какие паттерны ловит Холмс в бете?

В контексте нет информации о том, какие паттерны ловит Холмс в бете. Поэтому ответа не существует на основе предоставленного текста.


При ответе на первый вопрос модель сформулировала ответ непосредственно по найденным фрагментам. Во втором случае ответ в целом содержит нужную информацию, но модель при некоторых запусках слишком уверенно устанавливает причинно-следственную связь между найденными фактами, хотя в источнике она не сформулирована напрямую.

In [ ]:
def demo_rag(question, top_k=3, n_runs=1):
    context, results = build_context(question, top_k=top_k)

    print("=" * 80)
    print("ВОПРОС:")
    print(question)

    print("\nНАЙДЕННЫЕ ФРАГМЕНТЫ:")
    for result in results:
        print("-" * 80)
        print(f"{result['filename']} | score={result['score']:.3f}")
        print(result["text"])

    print("\n" + "=" * 80 )
    for i in range(n_runs):
        if n_runs > 1:            
            print(f"ЗАПУСК {i + 1}:")

        print(ask_rag(question, context=context))
        print("=" * 80)

In [97]:
demo_rag(test_questions[1], n_runs=3)

ВОПРОС:
Что произошло с Case-21 и почему он получил статус REVIEW?

НАЙДЕННЫЕ ФРАГМЕНТЫ:
--------------------------------------------------------------------------------
moriarty-tests.md | score=0.555
Holmes обнаруживает паттерн `RED`, но возвращает пустой `CaseReport`.
--------------------------------------------------------------------------------
case-archive.md | score=0.546
## Case-21 — Blue Window
Дата: 2025-03-17
Holmes обнаружил паттерн `BLUE`.
Watson сформировал объяснение, но не указал идентификатор источника.
Lestrade отклонил объяснение и перевёл дело в статус `REVIEW`.
--------------------------------------------------------------------------------
moriarty-tests.md | score=0.545
Ожидается, что Holmes обнаружит `BLUE`, а Lestrade переведёт случай в `REVIEW`.

ЗАПУСК 1:
Case-21 получил статус REVIEW потому что Lestrade отклонил объяснение, сформулированное Watsonом, после того как Holmes обнаружил паттерн `BLUE`.
ЗАПУСК 2:
Случай Case-21 был обнаружен Holmes, который обнар

#### Когда контекста недостаточно
Третий случай оказался особенно показательным. В нашей коллекции действительно есть ответ: в описании Beta 0.8 указано, что Holmes получил поддержку паттернов `RED` и `BLUE` и может автоматически обнаруживать эти типы аномалий. Но этот фрагмент не попал в результаты поиска для нашего запроса. Поэтому языковая модель не смогла воспользоваться информацией, которая фактически была в коллекции, и корректно сообщила, что в переданном контексте ответа нет. Это показывает важное различие: отсутствие ответа модели ещё не означает отсутствие нужной информации в нашей базе. Проблема может возникнуть раньше — на этапе поиска.

Модель не стала достраивать недостающую информацию самостоятельно и сообщила, что данных для ответа недостаточно.

Это важное свойство нашей постановки: качество ответа зависит не только от самой языковой модели, но и от того, какой контекст ей удалось предоставить. При этом инструкция не выдумывать ответ помогает контролировать ситуацию, когда найденной информации недостаточно.

Попробуем расширить контекст:

In [90]:
demo_rag(test_questions[-1], top_k=5)

ВОПРОС:
Какие паттерны ловит Холмс в бете?

НАЙДЕННЫЕ ФРАГМЕНТЫ:
--------------------------------------------------------------------------------
beta-release.md | score=0.613
После нескольких месяцев разработки команда 221B выпустила первую публичную бета-версию системы.
--------------------------------------------------------------------------------
overview.md | score=0.611
**Lestrade** проверяет результаты анализа и определяет, требует ли случай дополнительного рассмотрения.
--------------------------------------------------------------------------------
overview.md | score=0.590
**Moriarty** используется для создания контролируемых тестовых случаев, на которых проверяется работа системы.
--------------------------------------------------------------------------------
moriarty-tests.md | score=0.571
Moriarty используется для создания контролируемых случаев, позволяющих проверять отдельные этапы работы 221B.
---------------------------------------------------------------------------

Мы знаем, что ответ на этот вопрос есть в нашей коллекции: в описании Beta 0.8 прямо указано, какие паттерны поддерживает Holmes. Однако нужный фрагмент не попал в контекст даже после увеличения `top_k` до 5. Одна из возможных причин — формулировка запроса, в частности Холмс вместо Holmes. Эмбеддинговый поиск умеет сопоставлять тексты по смыслу, но такое сопоставление не обязательно одинаково хорошо работает для разных языков и формулировок.

Можно было бы попробовать расширить контекст ещё сильнее, но это не универсальное решение. Чем больше фрагментов мы передаём модели, тем больше среди них нерелевантной информации, а значит, модели становится сложнее выделить действительно важное. Кроме того, растёт объём контекста и стоимость обработки.

Вернёмся к этой проблеме, когда будем обсуждать способы улучшения поиска, а пока посмотрим на другой пример.

In [99]:
question = "Что изменилось в тестах GOLD 25 марта?"

demo_rag(question)

ВОПРОС:
Что изменилось в тестах GOLD 25 марта?

НАЙДЕННЫЕ ФРАГМЕНТЫ:
--------------------------------------------------------------------------------
beta-release.md | score=0.714
Поддержка `GOLD` пока находится в экспериментальном режиме и не включена в стандартный процесс расследования.
--------------------------------------------------------------------------------
moriarty-tests.md | score=0.702
Ожидается, что Holmes обнаружит `GOLD`. При этом стандартный процесс Beta 0.8 не должен автоматически передавать такой случай на проверку Lestrade.
--------------------------------------------------------------------------------
moriarty-tests.md | score=0.695
Сценарий содержит четыре последовательных события, необходимых для обнаружения `GOLD`.

В контексте не содержится информация о том, что изменилось в тестах GOLD 25 марта. Следовательно, на основе предоставленного контекста ответить на вопрос невозможно.


In [100]:
demo_rag(question, top_k=5, n_runs=5)

ВОПРОС:
Что изменилось в тестах GOLD 25 марта?

НАЙДЕННЫЕ ФРАГМЕНТЫ:
--------------------------------------------------------------------------------
beta-release.md | score=0.714
Поддержка `GOLD` пока находится в экспериментальном режиме и не включена в стандартный процесс расследования.
--------------------------------------------------------------------------------
moriarty-tests.md | score=0.702
Ожидается, что Holmes обнаружит `GOLD`. При этом стандартный процесс Beta 0.8 не должен автоматически передавать такой случай на проверку Lestrade.
--------------------------------------------------------------------------------
moriarty-tests.md | score=0.695
Сценарий содержит четыре последовательных события, необходимых для обнаружения `GOLD`.
--------------------------------------------------------------------------------
case-archive.md | score=0.642
## Case-24 — Gold Finch
Дата: 2025-03-25
Moriarty создал тестовый случай для экспериментального паттерна `GOLD`.
Holmes обнаружил паттерн 

#### Хороший контекст не гарантирует хороший ответ

В этом примере изменение `top_k` заметно меняет результат. При трёх найденных фрагментах информация о конкретном тестовом случае 25 марта не попадает в контекст, поэтому модель сообщает, что данных недостаточно. При `top_k=5` нужный фрагмент уже оказывается среди найденных результатов. Модель получает необходимую информацию и формирует ответ.

Однако даже имея нужный фрагмент, при разных запусках модель может повести себя по-разному: в одном случае дать вполне корректный ответ, в другом — оказаться слишком осторожной и отказаться от вывода, который можно было сделать из найденных фрагментов, а в третьем — связать их слишком свободно и добавить в ответ то, что из документов напрямую не следует.

Оба последних варианта могут быть проблемой. Если модель слишком осторожна, она не использует информацию, которая действительно есть в коллекции, но распределена по нескольким фрагментам. Если модель, наоборот, слишком свободно связывает найденные фрагменты, она может выдать правдоподобный вывод, который прямо не подтверждается документами.

Поэтому для практической системы недостаточно просто добавить поиск к языковой модели. Нужно контролировать и качество найденного контекста, и соответствие итогового ответа этому контексту.

#### Проверяем отсутствие ответа

Возьмём вопрос, для которого в нашей коллекции действительно нет нужной информации: документы сообщают, что Holmes обнаруживает `GOLD`, но ничего не говорят о том, какой алгоритм или внутренний механизм он для этого использует.

Сначала один раз посмотрим, какие фрагменты нашёл поиск. Затем зафиксируем этот контекст и несколько раз передадим его одной и той же языковой модели вместе с одним и тем же вопросом.

In [93]:
question = "Какой алгоритм использует Holmes для обнаружения GOLD?"

demo_rag(question, top_k=5, n_runs=5)

ВОПРОС:
Какой алгоритм использует Holmes для обнаружения GOLD?

НАЙДЕННЫЕ ФРАГМЕНТЫ:
--------------------------------------------------------------------------------
moriarty-tests.md | score=0.826
Ожидается, что Holmes обнаружит `GOLD`. При этом стандартный процесс Beta 0.8 не должен автоматически передавать такой случай на проверку Lestrade.
--------------------------------------------------------------------------------
beta-release.md | score=0.767
Поддержка `GOLD` пока находится в экспериментальном режиме и не включена в стандартный процесс расследования.
--------------------------------------------------------------------------------
moriarty-tests.md | score=0.760
Сценарий содержит четыре последовательных события, необходимых для обнаружения `GOLD`.
--------------------------------------------------------------------------------
overview.md | score=0.749
**Holmes** анализирует поток событий и обнаруживает подозрительные закономерности, которые могут указывать на инцидент.
------

Во всех пяти запусках модель отказалась придумывать алгоритм и явно указала, что такой информации в предоставленном контексте нет. Это хороший результат для нашего конкретного эксперимента: при одном и том же контексте модель стабильно следует заданному ограничению и не превращает упоминание `GOLD` в выдуманное описание алгоритма.

Но это не означает, что одной такой инструкции достаточно, чтобы гарантировать отсутствие фантазий в реальной системе. Мы уже видели обратный пример с Case-21: там модель связала несколько небольших фрагментов и сделала вывод, который не был полностью сформулирован в найденных документах.

Значит, контроль ответа остаётся отдельной задачей: нам нужно уметь проверять не только наличие релевантного контекста, но и то, насколько итоговый ответ действительно ему соответствует.

#### Проверяем разговорную формулировку

В предыдущих примерах мы в основном задавали вопросы достаточно близко к формулировкам документов. Но в реальной системе запрос может звучать иначе: человек не обязан использовать те же термины и конструкции, которые встречаются в коллекции.

Попробуем задать вопрос более естественно, как его мог бы сформулировать участник команды. В документах нет буквально такой формулировки. Посмотрим, сможет ли поиск найти фрагменты, из которых можно восстановить ответ.

In [140]:
demo_rag("Пофиксили empty CaseReport у Ватсона?", top_k=5)

ВОПРОС:
Пофиксили empty CaseReport у Ватсона?

НАЙДЕННЫЕ ФРАГМЕНТЫ:
--------------------------------------------------------------------------------
overview.md | score=0.592
**Holmes** анализирует поток событий и обнаруживает подозрительные закономерности, которые могут указывать на инцидент.
--------------------------------------------------------------------------------
beta-release.md | score=0.579
После нескольких месяцев разработки команда 221B выпустила первую публичную бета-версию системы.
--------------------------------------------------------------------------------
moriarty-tests.md | score=0.559
## Test Case D — Broken Report
Holmes обнаруживает паттерн `RED`, но возвращает пустой `CaseReport`. Сценарий используется для проверки поведения Watson при отсутствии данных для объяснения.
--------------------------------------------------------------------------------
git-history.txt | score=0.545
2026-03-18  fix Watson validation for empty CaseReport
Watson больше не формирует 

В данном случае разговорная формулировка не помешала ни поиску, ни LLM.

### Что можно улучшить и в каких направлениях развивать систему

Наш небольшой пример уже показывает основную идею RAG, но одновременно хорошо демонстрирует, почему в реальной системе одного поиска релевантных фрагментов и передачи их языковой модели недостаточно. Мы столкнулись сразу с несколькими проблемами:

- нужная информация может не попасть в top-k;
- качество поиска зависит от разбиения документов на фрагменты;
- нестандартная формулировка запроса может давать неожиданные результаты;
- модель может быть слишком осторожной или, наоборот, сделать вывод, который прямо не подтверждается источниками;
- наконец, сама коллекция документов может содержать информацию, которую мы вообще не хотим использовать.

Посмотрим, какие направления развития естественно следуют из этих наблюдений.

#### Подготовка документов

Первое направление — сделать сами фрагменты более информативными. В нашем демо каждый абзац стал отдельным чанком. Для реальной коллекции этого часто недостаточно: смысл абзаца может зависеть от заголовка, предыдущего текста или структуры документа. Можно сохранять вместе с текстом заголовки разделов, использовать перекрытие соседних фрагментов (overlap), учитывать размер и смысловые границы текста, добавлять метаданные. Тогда поиск будет работать не только с «голым» абзацем, но и с информацией о его месте и роли в документе.

#### Качество поиска

Мы используем эмбеддинги для поиска фрагментов, близких к вопросу по смыслу. Но качество такого поиска зависит от выбранной модели эмбеддингов. В нашем эксперименте это особенно заметно при работе с текстами на разных языках. Для русскоязычной системы может оказаться полезной мультиязычная модель эмбеддингов (multilingual embeddings), специально предназначенная для сопоставления текстов на разных языках.

Кроме того, эмбеддинги — не единственный способ поиска. В реальной системе можно использовать гибридный поиск (hybrid search), объединяющий семантический поиск с лексическим поиском, например BM25. Это особенно полезно для точных терминов, имён, идентификаторов, названий паттернов и других слов, которые важно найти буквально.

После первоначального поиска результаты можно дополнительно переупорядочить с помощью повторного ранжирования (reranking).

#### Количество передаваемых фрагментов

Параметр `top_k` влияет на результат, особенно когда вопрос сложный, требует информации из нескольких фрагментов или сформулирован так, что нужные фрагменты не имеют очевидного семантического совпадения с запросом. Если выбрать слишком маленькое значение, нужный фрагмент может вообще не попасть в контекст. Если слишком сильно увеличить `top_k`, контекст заполнится менее релевантной информацией, и модели станет сложнее выделить главное. Поэтому в реальной системе число передаваемых фрагментов не обязательно должно быть фиксированным. Его можно подбирать в зависимости от вопроса и результатов поиска, а сами найденные фрагменты — дополнительно фильтровать и переупорядочивать.

#### Работа с запросом пользователя

До сих пор мы предполагали, что пользователь уже сформулировал хороший вопрос. Но в реальной системе это далеко не всегда так. Запрос может быть слишком общим, разговорным или неоднозначным. Пользователь может использовать местоимения и ссылки на предыдущие сообщения диалога. Наконец, иногда пользователю вообще не нужен ответ на готовый вопрос: системе сначала нужно понять, что именно он хочет узнать, и сформулировать поисковый запрос самостоятельно.

Поэтому перед поиском можно добавить отдельный этап обработки запроса:
- переформулирование вопроса;
- выделение ключевых сущностей и ограничений;
- уточнение временного контекста;
- создание нескольких вариантов поискового запроса;
- при необходимости — запрос дополнительной информации у пользователя.

Такую обработку запроса часто называют переформулированием запроса (query rewriting) или расширением запроса (query expansion).

#### Повторный поиск

Если первый поиск не дал достаточно хорошего контекста, необязательно сразу отвечать пользователю, что данных недостаточно. Система может попробовать изменить запрос, расширить или сузить область поиска, использовать другой способ поиска или обратиться к другой коллекции документов. При этом повторный поиск можно запускать не только при полном отсутствии результатов. Система может оценивать качество найденного контекста и решать, достаточно ли его для ответа. Если нет, можно попробовать другой запрос или другую стратегию поиска.

#### Контроль сформированного ответа

Даже хороший контекст не гарантирует, что языковая модель аккуратно им воспользуется. Мы видели, что при разных запусках она может быть слишком осторожной и не использовать информацию, распределённую по нескольким фрагментам. В другом случае модель может, наоборот, слишком свободно связать найденные факты и сделать вывод, который прямо не подтверждается источниками. Поэтому в RAG важно контролировать не только качество поиска, но и то, насколько сформированный ответ действительно следует из предоставленного контекста. Один из способов сделать такой ответ более проверяемым — показывать пользователю, на каких источниках он основан. Вместо простого ответа модель может указывать документы или конкретные фрагменты, которые подтверждают отдельные утверждения. Это позволяет пользователю самостоятельно проверить ответ и даёт системе дополнительный материал для оценки его качества: можно проверить не только сам ответ, но и действительно ли указанные источники подтверждают сделанные в нём утверждения.

#### Оценка качества

До сих пор мы оценивали результаты в основном вручную. Но если система должна работать с большим количеством вопросов, этого недостаточно. Нам понадобится отдельная задача оценивания (evaluation) — автоматического или полуавтоматического определения качества результата.

Для RAG можно отдельно оценивать несколько этапов:
- насколько хорошо поиск находит нужные документы и фрагменты;
- достаточно ли найденного контекста для ответа;
- соответствует ли ответ найденным источникам;
- не добавляет ли модель неподтверждённые сведения;
- насколько полно и точно сформулирован ответ.

В зависимости от этапа можно использовать разные метрики. Например, для оценки поиска — полноту попадания нужных фрагментов в первые результаты, а для оценки ответов — соответствие источникам, полноту и точность ответа.

#### Качество самой коллекции

Не вся информация должна попадать в поиск. Мы уже видели пример с `april-fools.md`. Это файл, который формально является частью коллекции, но его содержимое не следует использовать как обычную техническую документацию. Документ может быть не только нерелевантным, но и устаревшим: более новая версия информации может уже находиться в другом файле. Это показывает более общую проблему: RAG сам по себе не умеет отличать полезный источник от нежелательного. Если документ попал в индекс, поиск может найти его.

Поэтому в реальной системе нужны дополнительные правила работы с коллекцией:
- отбор источников;
- удаление или исключение нерелевантных документов;
- учёт актуальности и версий;
- фильтрация по метаданным;
- контроль прав доступа (поиск должен учитывать не только релевантность документа, но и то, имеет ли пользователь право его видеть).


#### От RAG к более сложной архитектуре

Пока наш RAG — это фиксированная последовательность действий: вопрос → поиск → контекст → генерация ответа. Но рассмотренные улучшения постепенно делают этот процесс более динамичным. Например, система может решить, что найденного контекста недостаточно, переформулировать запрос и повторить поиск. Она может выбрать другую коллекцию, проверить полученный ответ, запросить уточнение у пользователя или отказаться от ответа, если подтверждений недостаточно.

Если система начинает самостоятельно выбирать такие действия в зависимости от промежуточных результатов, мы приходим к идее агентной архитектуры. В таком случае RAG становится не всей системой, а одним из инструментов, которыми может пользоваться агент.